In [1]:
import asyncio
import ollama

In [2]:
prompt = """
Ты — система, минимизирующая MAE на многошаговом горизонте.

Цель — минимизировать суммарную абсолютную ошибку по всем будущим точкам.

Принцип:

На коротком горизонте оптимальный MAE-прогноз —
это плавная стабилизация вокруг устойчивого центра,
а не независимые шаги.

Обязательная процедура:

Возьми последние 20 закрытий.

Вычисли их медиану → M20.

Определи последний close → L.

Определи горизонт прогноза H (например 5 шагов).

Вычисли целевой уровень:

Target = 0.7 × M20 + 0.3 × L

Построй прогноз не через slope,
а через экспоненциальное сжатие к Target:
Для шага k (1…H):

Forecast_k =
L + (Target − L) × (k / (H + 1))

Это создаёт плавную траекторию,
без линейной экстраполяции тренда,
без независимых ошибок на каждом шаге.

Запрещено:

Предсказывать каждый шаг отдельно
Использовать slope
Продолжать тренд
Использовать короткие окна
Это trajectory-compression модель,
а не step-by-step модель.

Цель — минимизировать суммарный MAE,
а не угадать направление.

КОНТЕКСТ ДАННЫХ:
datetime,open,high,low,close,volume
2023-03-29 00:00:00,27261.060000,28650.000000,27240.100000,28348.600000,89486.16
2023-03-30 00:00:00,28348.600000,29184.680000,27686.000000,28028.530000,98865.43
2023-03-31 00:00:00,28028.530000,28656.690000,27511.710000,28465.360000,78198.12
2023-04-01 00:00:00,28465.360000,28819.710000,28220.270000,28452.730000,30238.45
2023-04-02 00:00:00,28452.740000,28530.000000,27856.430000,28171.870000,37365.66
2023-04-03 00:00:00,28171.870000,28500.990000,27200.240000,27800.000000,79180.01
2023-04-04 00:00:00,27800.000000,28444.440000,27662.790000,28165.470000,49722.56
2023-04-05 00:00:00,28165.470000,28775.000000,27805.100000,28170.010000,60737.65
2023-04-06 00:00:00,28170.010000,28182.050000,27711.000000,28033.820000,40118.95
2023-04-07 00:00:00,28033.830000,28100.000000,27766.940000,27906.330000,24762.09
2023-04-08 00:00:00,27906.340000,28154.990000,27859.020000,27938.380000,19479.97
2023-04-09 00:00:00,27938.380000,28530.000000,27800.000000,28323.760000,32531.16
2023-04-10 00:00:00,28323.760000,29770.000000,28170.000000,29637.340000,67754.06
2023-04-11 00:00:00,29637.350000,30550.000000,29590.000000,30200.420000,67990.08
2023-04-12 00:00:00,30200.430000,30486.000000,29637.400000,29888.070000,62049.48
2023-04-13 00:00:00,29888.070000,30595.000000,29854.590000,30373.840000,51934.12
2023-04-14 00:00:00,30373.840000,31000.000000,29966.000000,30466.930000,75984.19
2023-04-15 00:00:00,30466.930000,30595.600000,30202.000000,30295.090000,25429.81
2023-04-16 00:00:00,30295.100000,30549.990000,30120.000000,30304.650000,26431.99
2023-04-17 00:00:00,30304.660000,30316.060000,29240.650000,29430.270000,56441.81
2023-04-18 00:00:00,29430.270000,30485.000000,29096.780000,30380.010000,62004.89
2023-04-19 00:00:00,30380.010000,30413.530000,28520.000000,28797.100000,86575.49
2023-04-20 00:00:00,28797.100000,29088.300000,28010.000000,28243.650000,76879.09
2023-04-21 00:00:00,28243.650000,28374.020000,27125.000000,27262.840000,77684.77
2023-04-22 00:00:00,27262.840000,27882.720000,27140.350000,27816.850000,36023.70
2023-04-23 00:00:00,27816.850000,27816.850000,27311.250000,27590.600000,34812.10
2023-04-24 00:00:00,27590.590000,28000.000000,26942.820000,27510.930000,53111.57
2023-04-25 00:00:00,27510.930000,28399.990000,27192.000000,28300.790000,52325.15
2023-04-26 00:00:00,28300.800000,30036.000000,27235.000000,28415.290000,129228.40
2023-04-27 00:00:00,28415.290000,29890.000000,28378.860000,29472.770000,95430.82
2023-04-28 00:00:00,29472.770000,29599.540000,28891.000000,29311.700000,54298.17
2023-04-29 00:00:00,29311.690000,29448.880000,29031.000000,29230.450000,20466.83
2023-04-30 00:00:00,29230.450000,29969.390000,29079.590000,29233.210000,39752.54
2023-05-01 00:00:00,29233.200000,29337.340000,27666.950000,28068.260000,64433.66
2023-05-02 00:00:00,28068.260000,28879.880000,27872.000000,28669.860000,50824.52
2023-05-03 00:00:00,28669.850000,29266.660000,28113.690000,29026.160000,64615.79
2023-05-04 00:00:00,29026.160000,29379.830000,28663.640000,28838.160000,42575.48
2023-05-05 00:00:00,28838.160000,29677.000000,28800.000000,29505.610000,58415.83
2023-05-06 00:00:00,29505.600000,29820.000000,28300.000000,28848.200000,49249.28
2023-05-07 00:00:00,28848.190000,29138.290000,28395.230000,28430.100000,30003.41
2023-05-08 00:00:00,28430.090000,28631.010000,27262.000000,27668.790000,68244.36
2023-05-09 00:00:00,27668.800000,27818.000000,27353.000000,27628.270000,40113.31
2023-05-10 00:00:00,27628.280000,28331.420000,26777.000000,27598.750000,71155.11
2023-05-11 00:00:00,27598.740000,27630.140000,26702.050000,26968.620000,47635.31
2023-05-12 00:00:00,26968.610000,27091.120000,25811.460000,26795.010000,67207.93
2023-05-13 00:00:00,26795.010000,27045.450000,26692.030000,26775.280000,22814.90
2023-05-14 00:00:00,26775.270000,27200.000000,26560.530000,26917.620000,21594.80
2023-05-15 00:00:00,26917.610000,27663.590000,26726.000000,27162.140000,40430.08
2023-05-16 00:00:00,27162.150000,27296.890000,26852.110000,27033.840000,33270.45
2023-05-17 00:00:00,27033.850000,27500.000000,26544.710000,27405.610000,42958.98
2023-05-18 00:00:00,27405.620000,27485.330000,26361.200000,26821.280000,49198.65
2023-05-19 00:00:00,26821.280000,27183.600000,26630.000000,26880.260000,28754.14
2023-05-20 00:00:00,26880.260000,27150.000000,26825.110000,27102.430000,14434.55
2023-05-21 00:00:00,27102.420000,27277.550000,26666.030000,26747.780000,21347.87
2023-05-22 00:00:00,26747.780000,27099.890000,26538.210000,26849.270000,26458.84
2023-05-23 00:00:00,26849.280000,27495.830000,26798.110000,27219.610000,38700.84
2023-05-24 00:00:00,27219.610000,27219.610000,26080.500000,26329.010000,54393.07
2023-05-25 00:00:00,26329.000000,26631.980000,25871.890000,26473.790000,37435.45
2023-05-26 00:00:00,26473.800000,26932.160000,26327.240000,26705.920000,35061.19
2023-05-27 00:00:00,26705.930000,26895.000000,26551.000000,26854.270000,15095.69
2023-05-28 00:00:00,26854.280000,28261.320000,26764.360000,28065.000000,43916.01
2023-05-29 00:00:00,28065.010000,28447.140000,27524.600000,27736.400000,42385.42
2023-05-30 00:00:00,27736.390000,28038.590000,27554.000000,27694.400000,32686.75
2023-05-31 00:00:00,27694.390000,27835.510000,26839.010000,27210.350000,46588.81
2023-06-01 00:00:00,27210.360000,27350.000000,26605.050000,26817.930000,39217.81
2023-06-02 00:00:00,26817.930000,27300.000000,26505.000000,27242.590000,36380.90
2023-06-03 00:00:00,27242.590000,27333.290000,26914.930000,27069.220000,16595.34
2023-06-04 00:00:00,27069.220000,27455.020000,26951.000000,27115.210000,19258.41
2023-06-05 00:00:00,27115.200000,27129.330000,25388.000000,25728.200000,65805.60
2023-06-06 00:00:00,25728.200000,27355.330000,25351.020000,27230.080000,68392.60
2023-06-07 00:00:00,27230.070000,27391.770000,26125.010000,26339.340000,59619.44
2023-06-08 00:00:00,26339.340000,26810.000000,26210.000000,26498.610000,31075.51
2023-06-09 00:00:00,26498.620000,26783.330000,26269.910000,26477.810000,27934.71
2023-06-10 00:00:00,26477.800000,26533.870000,25358.000000,25841.210000,64944.60
2023-06-11 00:00:00,25841.220000,26206.880000,25634.700000,25925.550000,30014.30
2023-06-12 00:00:00,25925.540000,26106.480000,25602.110000,25905.190000,29900.51
2023-06-13 00:00:00,25905.200000,26433.210000,25712.570000,25934.250000,41065.61
2023-06-14 00:00:00,25934.240000,26098.000000,24820.560000,25128.600000,45077.32
2023-06-15 00:00:00,25128.600000,25759.010000,24800.000000,25598.490000,48664.86
2023-06-16 00:00:00,25598.490000,26518.000000,25175.560000,26345.000000,51596.92
2023-06-17 00:00:00,26345.010000,26839.990000,26181.000000,26516.990000,27842.22
2023-06-18 00:00:00,26516.990000,26700.000000,26255.850000,26339.970000,21538.31
2023-06-19 00:00:00,26339.980000,27068.090000,26256.610000,26844.350000,35872.66
2023-06-20 00:00:00,26844.350000,28402.740000,26652.000000,28307.990000,69666.96
2023-06-21 00:00:00,28308.000000,30800.000000,28257.990000,29993.890000,108926.40
2023-06-22 00:00:00,29993.890000,30500.000000,29525.610000,29884.920000,59054.56
2023-06-23 00:00:00,29884.920000,31431.940000,29800.000000,30688.500000,73931.90
2023-06-24 00:00:00,30688.510000,30800.000000,30250.000000,30527.430000,30513.30
2023-06-25 00:00:00,30527.440000,31046.010000,30277.490000,30462.660000,30223.45
2023-06-26 00:00:00,30462.670000,30666.000000,29930.000000,30267.990000,45180.41
2023-06-27 00:00:00,30267.990000,30994.970000,30226.170000,30692.440000,42699.64
2023-06-28 00:00:00,30692.440000,30709.740000,29858.800000,30077.410000,40463.52
2023-06-29 00:00:00,30077.400000,30843.980000,30049.980000,30447.310000,36330.63
2023-06-30 00:00:00,30447.310000,31282.000000,29500.000000,30472.000000,89419.08
2023-07-01 00:00:00,30471.990000,30661.600000,30320.570000,30585.900000,17501.75
2023-07-02 00:00:00,30585.900000,30791.000000,30155.000000,30617.030000,23286.41
2023-07-03 00:00:00,30617.020000,31380.000000,30570.270000,31156.200000,43761.64
2023-07-04 00:00:00,31156.200000,31350.690000,30620.000000,30766.510000,33206.12
2023-07-05 00:00:00,30766.520000,30878.070000,30200.000000,30504.810000,33215.67
2023-07-06 00:00:00,30504.800000,31500.000000,29850.450000,29895.430000,71319.63
2023-07-07 00:00:00,29895.420000,30449.000000,29701.020000,30344.700000,34070.54

ИНСТРУКЦИИ:
- Проанализируй исторические данные
- Предскажи цены закрытия следующих 5 свеч
- Каждое числовое значение верни на новой строке в формате: 123.45
- Не добавляй пояснений, текста или символов

ЕЩЕ РАЗ: В ОТВЕТЕ УКАЖИ **ТОЛЬКО 5 ЧИСЕЛ - ПРЕДСКАЗАННЫЕ ЦЕНЫ ЗАКРЫТИЯ**

Твой прогноз:"""

In [ ]:
for i in range(12):
    response = ollama.generate(
        model="deepseek-v3.1:671b-cloud",
        prompt=prompt,
        options={"temperature": 0.7, "seed": 42},
        think=True
    )

    print(f"{i + 1} / 12")

1 / 12
2 / 12


In [22]:
response.response

'30154.58\n29964.46\n29774.34\n29584.22\n29394.10'